# DeepMzyme Training Notebook

This notebook builds and optionally runs user-selected DeepMzyme training configurations in Google Colab. It uses validation metrics for model selection and keeps held-out test evaluation as a separate final step.

The default plan is a dry run: one 1-epoch Only-GVP metal configuration with radius-only graph edges and the full node feature set. It does not require ESM embeddings, RING files, a RING executable, Google Drive, or held-out test evaluation.

## Recommended First Runs

1. 1-epoch Only-GVP radius-only dry/smoke run.
2. Only-GVP radius-only small learning-rate/seed comparison.
3. Only-GVP radius-only feature-ablation check, for example full features vs omit `v_cb_to_fg`.
4. Only-GVP with precomputed RING compared against radius-only.
5. ESM/fusion models only after ESM embeddings are confirmed.

In [ ]:
#@title Runtime/environment checks
import os
import platform
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
NOTEBOOK_START_CWD = Path.cwd()

print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("Current working directory:", NOTEBOOK_START_CWD)
print("Running in Colab-like runtime:", IN_COLAB)

if not IN_COLAB:
    print("Local runtime detected. In the project environment, the expected interpreter is:")
    print("/home/mechti/miniconda3/envs/DeepMzyme/bin/python")

In [ ]:
#@title Main configuration
# A. Basic controls
TASK = "metal"  #@param ["metal", "ec", "joint"]
EPOCHS = 1  #@param {type:"integer"}
RUN_TRAINING = False  #@param {type:"boolean"}
DEVICE = "cpu"  #@param ["cpu", "cuda"]
RUN_HELD_OUT_TEST_EVAL = False  #@param {type:"boolean"}

# B. Multiple configuration examination controls
# MODEL_PRESETS_CSV accepts: Only-GVP, Only-ESM, GVP + late fusion, GVP + early fusion, GVP + node-level late fusion, GVP + hybrid fusion, GVP + cross-modal attention, SimpleGNN + ESM
MODEL_PRESETS_CSV = "Only-GVP"  #@param {type:"string"}
# RING_EDGE_MODES_CSV accepts: radius_only, radius_plus_precomputed_ring, generate_missing_ring
RING_EDGE_MODES_CSV = "radius_only"  #@param {type:"string"}
BATCH_SIZES_CSV = "4"  #@param {type:"string"}
LEARNING_RATES_CSV = "3e-5"  #@param {type:"string"}
WEIGHT_DECAYS_CSV = "1e-4"  #@param {type:"string"}
SEEDS_CSV = "42"  #@param {type:"string"}
MAX_CONFIGURATION_RUNS = 24  #@param {type:"integer"}
STOP_ON_FIRST_FAILURE = True  #@param {type:"boolean"}
SKIP_EXISTING_RUNS = True  #@param {type:"boolean"}

# C. Data controls
COLAB_DATA_SOURCE = "huggingface_link"  #@param ["huggingface_link", "upload_file", "drive"]
REPO_GIT_URL = "https://github.com/MECHTI1/DeepMzyme.git"  #@param {type:"string"}
REPO_GIT_BRANCH = "main"  #@param {type:"string"}
REPO_ROOT = ""  #@param {type:"string"}
INSTALL_REQUIREMENTS = True  #@param {type:"boolean"}
INSTALL_ESM_DEPENDENCIES = False  #@param {type:"boolean"}
MOUNT_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/DeepMzyme"  #@param {type:"string"}
# Do not modify BUNDLE_FILENAME, BUNDLE_URL, or BUNDLE_SHA256 unless using a custom bundle.
BUNDLE_FILENAME = "DeepMzyme_Data_runtime_local_2026-05-03.tar.zst"  #@param {type:"string"}
BUNDLE_URL = "https://huggingface.co/datasets/GMBioinformatics/DeepMzyme/resolve/main/DeepMzyme_Data_runtime_local_2026-05-03.tar.zst"  #@param {type:"string"}
BUNDLE_SHA256 = "c86faa40ff69c021de02b72b5fef9ebd1712f5ef8e6cb3da27b3a9e8261816c1"  #@param {type:"string"}
DATASET_NAME = "train_and_test_sets_structures_non_overlapped_pinmymetal"  #@param {type:"string"}
TRAIN_DIR_OVERRIDE = ""  #@param {type:"string"}
TRAIN_CSV_OVERRIDE = ""  #@param {type:"string"}
TEST_DIR_OVERRIDE = ""  #@param {type:"string"}
TEST_CSV_OVERRIDE = ""  #@param {type:"string"}

# D. ESM controls
ESM_EMBEDDINGS_DIR = ""  #@param {type:"string"}
PREPARE_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
DISABLE_ESM_BRANCH = False  #@param {type:"boolean"}
ESM_DIM = 960  #@param {type:"integer"}

# E. RING controls
RING_FEATURES_DIR = ""  #@param {type:"string"}
RING_EXE_PATH = "DeepMzyme_Data/ring-4.0/out/bin/ring"  #@param {type:"string"}

# F. Node-feature ablation controls
# Blank means full node feature set. Use semicolons between omission configurations; commas inside one entry omit features together.
# Example: ;v_cb_to_fg;v_cb_to_fg,v_res_to_metal plans full features, one omitted feature, then two features omitted together.
OMIT_NODE_FEATURE_SETS = ""  #@param {type:"string"}

# G. Advanced model/training controls
VAL_FRACTION = 0.15  #@param {type:"number"}
SPLIT_BY = "pdbid"  #@param ["pdbid", "pdbid_chain", "structure_id", "pocket_id"]
# Leave blank to use the recommended default for the selected task
# (val_metal_balanced_acc for metal, val_ec_group_balanced_acc for ec, val_joint_balanced_acc for joint).
SELECTION_METRIC = ""  #@param {type:"string"}
HIDDEN_S_VALUES_CSV = "128"  #@param {type:"string"}
HIDDEN_V_VALUES_CSV = "16"  #@param {type:"string"}
EDGE_HIDDEN_VALUES_CSV = "64"  #@param {type:"string"}
GVP_LAYERS_VALUES_CSV = "4"  #@param {type:"string"}
HEAD_MLP_LAYERS_VALUES_CSV = "2"  #@param {type:"string"}
EDGE_RADIUS_VALUES_CSV = "8.0"  #@param {type:"string"}
ESM_FUSION_DIM_VALUES_CSV = "128"  #@param {type:"string"}
LR_SCHEDULES_CSV = "fixed"  #@param {type:"string"}
LR_STEP_SIZE = 10  #@param {type:"integer"}
LR_DECAY_GAMMA = 0.5  #@param {type:"number"}
NODE_RBF_SIGMA = 0.75  #@param {type:"number"}
EDGE_RBF_SIGMA = 0.75  #@param {type:"number"}
NODE_RBF_USE_RAW_DISTANCES = False  #@param {type:"boolean"}
EARLY_ESM_DIM = 32  #@param {type:"integer"}
EARLY_ESM_DROPOUT = 0.2  #@param {type:"number"}
EARLY_ESM_RAW = False  #@param {type:"boolean"}
EARLY_ESM_SCOPE = "all"  #@param ["all", "first_shell", "first_second_shell"]
CROSS_ATTENTION_LAYERS_CSV = "1"  #@param {type:"string"}
CROSS_ATTENTION_HEADS_CSV = "4"  #@param {type:"string"}
CROSS_ATTENTION_DROPOUT = 0.1  #@param {type:"number"}
CROSS_ATTENTION_NEIGHBORHOOD = "all"  #@param ["all", "first_shell", "first_second_shell"]
CROSS_ATTENTION_BIDIRECTIONAL = False  #@param {type:"boolean"}
EC_LABEL_DEPTHS_CSV = "1"  #@param {type:"string"}
EC_GROUP_WEIGHTING = "structure_id"  #@param ["none", "structure_id", "pdbid_chain", "pdbid"]
EC_CONTRASTIVE_WEIGHTS_CSV = "0.0"  #@param {type:"string"}
EC_CONTRASTIVE_TEMPERATURE = 0.1  #@param {type:"number"}
METAL_LOSS_FUNCTION = "cross_entropy"  #@param ["cross_entropy", "focal"]
METAL_FOCAL_GAMMA = 2.0  #@param {type:"number"}
METAL_LABEL_SMOOTHING = 0.0  #@param {type:"number"}
METAL_LOSS_WEIGHT = 1.0  #@param {type:"number"}
EC_LOSS_WEIGHT = 1.0  #@param {type:"number"}
MN_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
CU_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
ZN_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
FE_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
CO_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
NI_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
CLASS_VIII_LOSS_MULTIPLIER = 1.0  #@param {type:"number"}
BALANCE_METAL_SITE_SYMBOLS = False  #@param {type:"boolean"}
REQUIRE_ALL_TASK_CLASSES = False  #@param {type:"boolean"}
ALLOW_MISSING_EXTERNAL_FEATURES = True  #@param {type:"boolean"}
EXTERNAL_FEATURES_ROOT_DIR = ""  #@param {type:"string"}
EXTERNAL_FEATURE_SOURCE = "auto"  #@param ["auto", "bluues_rosetta", "updated"]
N_FOLDS = ""  #@param {type:"string"}
FOLD_INDEX = ""  #@param {type:"string"}
DETERMINISTIC = False  #@param {type:"boolean"}
SAVE_EPOCH_CHECKPOINTS = False  #@param {type:"boolean"}
ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG = False  #@param {type:"boolean"}
UNSUPPORTED_METAL_POLICY = "error"  #@param ["error", "skip"]
INVALID_STRUCTURE_POLICY = "skip"  #@param ["error", "skip"]

# H. Output/reporting controls
# Colab note: the default output path is ephemeral and lost when the session resets.
# For multi-epoch runs, set RUNS_DIR to a Drive path or enable COPY_OUTPUTS_TO_DRIVE.
RUNS_DIR = ""  #@param {type:"string"}
RUN_NAME_PREFIX = ""  #@param {type:"string"}
COPY_OUTPUTS_TO_DRIVE = False  #@param {type:"boolean"}
SUMMARY_BASENAME = "deepmzyme_configuration_comparison"  #@param {type:"string"}

print("Main configuration captured. Run the next cell to build CONFIG.")

In [ ]:
#@title Build central CONFIG dictionary
import json
from pathlib import Path

CONFIG = {
    "basic": {
        "task": TASK,
        "epochs": int(EPOCHS),
        "run_training": bool(RUN_TRAINING),
        "device": DEVICE,
        "run_held_out_test_eval": bool(RUN_HELD_OUT_TEST_EVAL),
    },
    "configuration_comparison": {
        "model_presets_csv": MODEL_PRESETS_CSV,
        "ring_edge_modes_csv": RING_EDGE_MODES_CSV,
        "batch_sizes_csv": BATCH_SIZES_CSV,
        "learning_rates_csv": LEARNING_RATES_CSV,
        "weight_decays_csv": WEIGHT_DECAYS_CSV,
        "seeds_csv": SEEDS_CSV,
        "max_configuration_runs": int(MAX_CONFIGURATION_RUNS),
        "stop_on_first_failure": bool(STOP_ON_FIRST_FAILURE),
        "skip_existing_runs": bool(SKIP_EXISTING_RUNS),
    },
    "data": {
        "colab_data_source": COLAB_DATA_SOURCE,
        "repo_git_url": REPO_GIT_URL,
        "repo_git_branch": REPO_GIT_BRANCH,
        "repo_root": REPO_ROOT,
        "install_requirements": bool(INSTALL_REQUIREMENTS),
        "install_esm_dependencies": bool(INSTALL_ESM_DEPENDENCIES),
        "mount_drive": bool(MOUNT_DRIVE),
        "drive_root": DRIVE_ROOT,
        "bundle_filename": BUNDLE_FILENAME,
        "bundle_url": BUNDLE_URL,
        "bundle_sha256": BUNDLE_SHA256,
        "dataset_name": DATASET_NAME,
        "train_dir_override": TRAIN_DIR_OVERRIDE,
        "train_csv_override": TRAIN_CSV_OVERRIDE,
        "test_dir_override": TEST_DIR_OVERRIDE,
        "test_csv_override": TEST_CSV_OVERRIDE,
    },
    "esm": {
        "esm_embeddings_dir": ESM_EMBEDDINGS_DIR,
        "prepare_missing_esm_embeddings": bool(PREPARE_MISSING_ESM_EMBEDDINGS),
        "allow_missing_esm_embeddings": bool(ALLOW_MISSING_ESM_EMBEDDINGS),
        "disable_esm_branch": bool(DISABLE_ESM_BRANCH),
        "esm_dim": int(ESM_DIM),
    },
    "ring": {
        "ring_features_dir": RING_FEATURES_DIR,
        "ring_exe_path": RING_EXE_PATH,
    },
    "node_features": {
        "node_feature_set": "conservative",
        "omit_node_feature_sets": OMIT_NODE_FEATURE_SETS,
    },
    "advanced": {
        "val_fraction": float(VAL_FRACTION),
        "split_by": SPLIT_BY,
        "selection_metric": SELECTION_METRIC,
        "hidden_s_values_csv": HIDDEN_S_VALUES_CSV,
        "hidden_v_values_csv": HIDDEN_V_VALUES_CSV,
        "edge_hidden_values_csv": EDGE_HIDDEN_VALUES_CSV,
        "gvp_layers_values_csv": GVP_LAYERS_VALUES_CSV,
        "head_mlp_layers_values_csv": HEAD_MLP_LAYERS_VALUES_CSV,
        "edge_radius_values_csv": EDGE_RADIUS_VALUES_CSV,
        "esm_fusion_dim_values_csv": ESM_FUSION_DIM_VALUES_CSV,
        "lr_schedules_csv": LR_SCHEDULES_CSV,
        "lr_step_size": int(LR_STEP_SIZE),
        "lr_decay_gamma": float(LR_DECAY_GAMMA),
        "node_rbf_sigma": float(NODE_RBF_SIGMA),
        "edge_rbf_sigma": float(EDGE_RBF_SIGMA),
        "node_rbf_use_raw_distances": bool(NODE_RBF_USE_RAW_DISTANCES),
        "early_esm_dim": int(EARLY_ESM_DIM),
        "early_esm_dropout": float(EARLY_ESM_DROPOUT),
        "early_esm_raw": bool(EARLY_ESM_RAW),
        "early_esm_scope": EARLY_ESM_SCOPE,
        "cross_attention_layers_csv": CROSS_ATTENTION_LAYERS_CSV,
        "cross_attention_heads_csv": CROSS_ATTENTION_HEADS_CSV,
        "cross_attention_dropout": float(CROSS_ATTENTION_DROPOUT),
        "cross_attention_neighborhood": CROSS_ATTENTION_NEIGHBORHOOD,
        "cross_attention_bidirectional": bool(CROSS_ATTENTION_BIDIRECTIONAL),
        "ec_label_depths_csv": EC_LABEL_DEPTHS_CSV,
        "ec_group_weighting": EC_GROUP_WEIGHTING,
        "ec_contrastive_weights_csv": EC_CONTRASTIVE_WEIGHTS_CSV,
        "ec_contrastive_temperature": float(EC_CONTRASTIVE_TEMPERATURE),
        "metal_loss_function": METAL_LOSS_FUNCTION,
        "metal_focal_gamma": float(METAL_FOCAL_GAMMA),
        "metal_label_smoothing": float(METAL_LABEL_SMOOTHING),
        "metal_loss_weight": float(METAL_LOSS_WEIGHT),
        "ec_loss_weight": float(EC_LOSS_WEIGHT),
        "mn_loss_multiplier": float(MN_LOSS_MULTIPLIER),
        "cu_loss_multiplier": float(CU_LOSS_MULTIPLIER),
        "zn_loss_multiplier": float(ZN_LOSS_MULTIPLIER),
        "fe_loss_multiplier": float(FE_LOSS_MULTIPLIER),
        "co_loss_multiplier": float(CO_LOSS_MULTIPLIER),
        "ni_loss_multiplier": float(NI_LOSS_MULTIPLIER),
        "class_viii_loss_multiplier": float(CLASS_VIII_LOSS_MULTIPLIER),
        "balance_metal_site_symbols": bool(BALANCE_METAL_SITE_SYMBOLS),
        "require_all_task_classes": bool(REQUIRE_ALL_TASK_CLASSES),
        "allow_missing_external_features": bool(ALLOW_MISSING_EXTERNAL_FEATURES),
        "external_features_root_dir": EXTERNAL_FEATURES_ROOT_DIR,
        "external_feature_source": EXTERNAL_FEATURE_SOURCE,
        "n_folds": N_FOLDS,
        "fold_index": FOLD_INDEX,
        "deterministic": bool(DETERMINISTIC),
        "save_epoch_checkpoints": bool(SAVE_EPOCH_CHECKPOINTS),
        "allow_train_loss_test_eval_debug": bool(ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG),
        "unsupported_metal_policy": UNSUPPORTED_METAL_POLICY,
        "invalid_structure_policy": INVALID_STRUCTURE_POLICY,
    },
    "output": {
        "runs_dir": RUNS_DIR,
        "run_name_prefix": RUN_NAME_PREFIX,
        "copy_outputs_to_drive": bool(COPY_OUTPUTS_TO_DRIVE),
        "summary_basename": SUMMARY_BASENAME,
    },
}

print(json.dumps(CONFIG, indent=2, sort_keys=True))

## Configuration Option Reference

CSV fields accept comma-separated values. The notebook builds one planned configuration for each selected combination, capped by `MAX_CONFIGURATION_RUNS`.

**Common controls**

| Field | Allowed values or format | Example |
|---|---|---|
| `TASK` | `metal`, `ec`, `joint` | `metal` |
| `MODEL_PRESETS_CSV` | `Only-GVP`, `Only-ESM`, `GVP + late fusion`, `GVP + early fusion`, `GVP + node-level late fusion`, `GVP + hybrid fusion`, `GVP + cross-modal attention`, `SimpleGNN + ESM` | `Only-GVP,GVP + late fusion` |
| `RING_EDGE_MODES_CSV` | `radius_only`, `radius_plus_precomputed_ring`, `generate_missing_ring` | `radius_only,radius_plus_precomputed_ring` |
| `BATCH_SIZES_CSV` | comma-separated integers | `4,8` |
| `LEARNING_RATES_CSV` | comma-separated numbers | `1e-4,3e-5` |
| `WEIGHT_DECAYS_CSV` | comma-separated numbers | `1e-4,0` |
| `SEEDS_CSV` | comma-separated integers | `42,43,44` |
| `SPLIT_BY` | `pdbid`, `pdbid_chain`, `structure_id`, `pocket_id` | `pdbid` |
| `SELECTION_METRIC` | validation metric accepted by `src/train.py`; recommended defaults are `val_metal_balanced_acc`, `val_ec_group_balanced_acc`, `val_joint_balanced_acc` | `val_metal_balanced_acc` |
| `LR_SCHEDULES_CSV` | `fixed`, `cosine`, `step` | `fixed` |

**Model preset mapping**

| Preset | CLI mapping | Needs ESM embeddings |
|---|---|---|
| `Only-GVP` | `--model-architecture only_gvp` | No |
| `Only-ESM` | `--model-architecture only_esm` | Yes |
| `GVP + late fusion` | `--model-architecture gvp --fusion-mode late_fusion` | Yes |
| `GVP + early fusion` | `--model-architecture gvp --fusion-mode early_fusion --use-early-esm` | Yes |
| `GVP + node-level late fusion` | `--model-architecture gvp --fusion-mode node_level_late_fusion` | Yes |
| `GVP + hybrid fusion` | `--model-architecture gvp --fusion-mode hybrid --use-early-esm` | Yes |
| `GVP + cross-modal attention` | `--model-architecture gvp --fusion-mode cross_modal_attention` plus cross-attention controls | Yes |
| `SimpleGNN + ESM` | `--model-architecture simple_gnn_esm --fusion-mode late_fusion` | Yes |

**RING modes**

| Mode | Behavior |
|---|---|
| `radius_only` | Radius graph edges only. No RING files or executable required. |
| `radius_plus_precomputed_ring` | Adds existing RING edge files. Requires `RING_FEATURES_DIR` when training is enabled. |
| `generate_missing_ring` | Adds RING edges and generates missing files. Requires executable `RING_EXE_PATH` when training is enabled. |

**Node-feature omission**

`OMIT_NODE_FEATURE_SETS` uses semicolons between planned omission settings. Commas inside one setting mean those features are omitted together.

Valid feature names are discovered from the repository and printed by the command-building cell. Current conservative names are:

`aa_one_hot`, `hydrophobicity_kd`, `donor_flag`, `acceptor_flag`, `aromatic_flag`, `acidic_flag`, `basic_flag`, `is_first_shell`, `is_second_shell`, `ca_to_metal`, `fg_to_metal`, `min_donor_to_metal`, `biotite_residue_sasa`, `custom_charge_distance_proxy`, `dpka_titr`, `v_cb_to_fg`, `v_res_to_metal`, `cos_theta_between_vnetligand_to_vrestometal`.

Examples:

| Value | Meaning |
|---|---|
| blank | full node feature set |
| `v_cb_to_fg` | one configuration omitting `v_cb_to_fg` |
| `;v_cb_to_fg;v_cb_to_fg,v_res_to_metal` | full features, then omit `v_cb_to_fg`, then omit both listed vector geometry features |

**Advanced allowed values**

| Field | Allowed values or format |
|---|---|
| `COLAB_DATA_SOURCE` | `huggingface_link`, `upload_file`, `drive` |
| `EXTERNAL_FEATURE_SOURCE` | `auto`, `bluues_rosetta`, `updated` |
| `EARLY_ESM_SCOPE` | `all`, `first_shell`, `first_second_shell` |
| `CROSS_ATTENTION_NEIGHBORHOOD` | `all`, `first_shell`, `first_second_shell` |
| `EC_GROUP_WEIGHTING` | `none`, `structure_id`, `pdbid_chain`, `pdbid` |
| `METAL_LOSS_FUNCTION` | `cross_entropy`, `focal` |
| `UNSUPPORTED_METAL_POLICY` | `error`, `skip` |
| `INVALID_STRUCTURE_POLICY` | `error`, `skip` |
| Capacity CSV fields | comma-separated numbers or integers: `HIDDEN_S_VALUES_CSV`, `HIDDEN_V_VALUES_CSV`, `EDGE_HIDDEN_VALUES_CSV`, `GVP_LAYERS_VALUES_CSV`, `HEAD_MLP_LAYERS_VALUES_CSV`, `EDGE_RADIUS_VALUES_CSV`, `ESM_FUSION_DIM_VALUES_CSV`, `CROSS_ATTENTION_LAYERS_CSV`, `CROSS_ATTENTION_HEADS_CSV`, `EC_LABEL_DEPTHS_CSV`, `EC_CONTRASTIVE_WEIGHTS_CSV` |

Every command-line flag generated by this notebook is checked against `src/train.py`; unsupported flags fail before commands are shown.

In [ ]:
#@title Clone or locate DeepMzyme repository and install dependencies
import os
import shutil
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    return "google.colab" in sys.modules or Path("/content").exists()


def path_has_repo(path: Path) -> bool:
    return (path / "src" / "train.py").is_file() and (path / "Plan.md").is_file()


def resolve_repo_dir() -> Path:
    configured = str(CONFIG["data"]["repo_root"] or "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if path_has_repo(candidate):
            return candidate.resolve()
    default_colab = Path("/content/DeepMzyme")
    if path_has_repo(default_colab):
        return default_colab.resolve()
    return default_colab


REPO_DIR = resolve_repo_dir()
if path_has_repo(REPO_DIR):
    print("Using repository:", REPO_DIR)
else:
    if REPO_DIR.exists():
        raise FileNotFoundError(f"Configured repository path exists but is not a DeepMzyme checkout: {REPO_DIR}")
    print("Cloning repository to:", REPO_DIR)
    subprocess.run(
        ["git", "clone", "--branch", CONFIG["data"]["repo_git_branch"], CONFIG["data"]["repo_git_url"], str(REPO_DIR)],
        check=True,
    )

for cmd in (["git", "branch", "--show-current"], ["git", "rev-parse", "HEAD"], ["git", "status", "--short"]):
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print("$", " ".join(cmd))
    print(result.stdout.strip() or result.stderr.strip() or "(no output)")

requirements_path = REPO_DIR / "src" / "requirements.txt"
if CONFIG["data"]["install_requirements"]:
    if requirements_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], check=True)
    else:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "torch-geometric==2.7.0",
                "biopython==1.87",
                "biotite>=1.6",
                "propka>=3.5",
                "gemmi>=0.7",
                "pandas",
                "matplotlib",
            ],
            check=True,
        )
else:
    print("Dependency installation skipped.")

if CONFIG["data"]["install_esm_dependencies"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "esm"], check=True)
else:
    print("Optional ESM dependency installation skipped.")

SRC_DIR = REPO_DIR / "src"
os.environ["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Training entry point:", REPO_DIR / "src" / "train.py")
print("PYTHONPATH starts with:", os.environ["PYTHONPATH"].split(os.pathsep)[0])

In [ ]:
#@title Data source configuration and bundle/data setup
import hashlib
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path("/tmp/deepmzyme_colab_runtime")
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT_PATH = Path(CONFIG["data"]["drive_root"]).expanduser()
DRIVE_DATA_DIR = DRIVE_ROOT_PATH / "DeepMzyme_Data"
LOCAL_BUNDLE_DIR = RUNTIME_ROOT / "deepmzyme_bundle"
LOCAL_DOWNLOAD_PATH = RUNTIME_ROOT / CONFIG["data"]["bundle_filename"]
LOCAL_DATA_DIR = RUNTIME_ROOT / "deepmzyme_data"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_sha256(path: Path, expected: str) -> None:
    expected = str(expected or "").strip().lower()
    if not expected:
        print("No SHA256 provided; skipping checksum verification for:", path)
        return
    observed = sha256_file(path).lower()
    if observed != expected:
        raise ValueError(f"SHA256 mismatch for {path}: expected {expected}, observed {observed}")
    print("Verified bundle SHA256:", observed)


def ensure_zstd_available() -> None:
    if shutil.which("zstd") is not None:
        return
    if Path("/content").exists():
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
        return
    raise RuntimeError("zstd is required to unpack .tar.zst bundles but was not found on PATH.")


def unpack_bundle(bundle_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / ".deepmzyme_bundle_unpacked"
    if marker.exists():
        print("Bundle already unpacked:", destination)
        return
    if str(bundle_path).endswith(".tar.zst"):
        ensure_zstd_available()
        subprocess.run(["tar", "--use-compress-program=zstd", "-xf", str(bundle_path), "-C", str(destination)], check=True)
    elif str(bundle_path).endswith((".tar.gz", ".tgz", ".tar")):
        subprocess.run(["tar", "-xf", str(bundle_path), "-C", str(destination)], check=True)
    else:
        raise ValueError(f"Unsupported bundle format: {bundle_path}")
    marker.write_text("ok\n", encoding="utf-8")
    print("Unpacked bundle to:", destination)


if (CONFIG["data"]["mount_drive"] or CONFIG["data"]["colab_data_source"] == "drive") and Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT_CANDIDATES = []
source = CONFIG["data"]["colab_data_source"]
if source == "huggingface_link":
    if not LOCAL_DOWNLOAD_PATH.exists():
        print("Downloading:", CONFIG["data"]["bundle_url"])
        urllib.request.urlretrieve(CONFIG["data"]["bundle_url"], LOCAL_DOWNLOAD_PATH)
    else:
        print("Using existing downloaded bundle:", LOCAL_DOWNLOAD_PATH)
    verify_sha256(LOCAL_DOWNLOAD_PATH, CONFIG["data"]["bundle_sha256"])
    unpack_bundle(LOCAL_DOWNLOAD_PATH, LOCAL_BUNDLE_DIR)
    DATA_ROOT_CANDIDATES.extend([LOCAL_BUNDLE_DIR, LOCAL_DATA_DIR, REPO_DIR / "DeepMzyme_Data"])
elif source == "upload_file":
    if not Path("/content").exists():
        raise RuntimeError("upload_file mode is available only in Colab.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No bundle file was uploaded.")
    uploaded_name = next(iter(uploaded))
    bundle_path = Path("/content") / uploaded_name
    verify_sha256(bundle_path, CONFIG["data"]["bundle_sha256"])
    unpack_bundle(bundle_path, LOCAL_BUNDLE_DIR)
    DATA_ROOT_CANDIDATES.extend([LOCAL_BUNDLE_DIR, LOCAL_DATA_DIR, REPO_DIR / "DeepMzyme_Data"])
elif source == "drive":
    DATA_ROOT_CANDIDATES.extend([DRIVE_DATA_DIR, LOCAL_BUNDLE_DIR, LOCAL_DATA_DIR, REPO_DIR / "DeepMzyme_Data"])
else:
    raise ValueError("COLAB_DATA_SOURCE must be 'huggingface_link', 'upload_file', or 'drive'.")

print("Data source:", source)
print("Data root candidates:")
for candidate in DATA_ROOT_CANDIDATES:
    print(" ", candidate, "exists:", Path(candidate).exists())

In [ ]:
#@title Detect dataset paths and validate inputs
import csv
from pathlib import Path

SITE_SUMMARY_COLUMN_ALIASES = {
    "pdbid": {"pdbid", "structure"},
    "metal residue number": {"metal residue number", "chain_resi"},
    "EC number": {"ec number", "ecnumber"},
    "metal residue type": {"metal residue type", "metaltype"},
}
STRUCTURE_INSPECTION_COLUMNS = {"structure_name", "metal_type"}
STRUCTURE_EXTENSIONS = ("*.pdb", "*.cif", "*.mmcif")


def csv_fieldnames(path: Path) -> set[str]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        return {field.strip().lower() for field in (reader.fieldnames or []) if field}


def classify_summary_csv(path: Path) -> str:
    fields = csv_fieldnames(path)
    has_site_columns = all(fields.intersection(options) for options in SITE_SUMMARY_COLUMN_ALIASES.values())
    if has_site_columns:
        return "site-level MAHOMES summary CSV"
    if STRUCTURE_INSPECTION_COLUMNS.issubset(fields):
        return "structure-level inspection CSV"
    return "unrecognized CSV"


def is_site_summary_csv(path: Path) -> bool:
    try:
        return classify_summary_csv(path) == "site-level MAHOMES summary CSV"
    except Exception:
        return False


def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return sum(1 for _ in csv.DictReader(handle))


def find_dataset_root(root: Path, dataset_name: str) -> Path | None:
    root = Path(root)
    candidates = [root / dataset_name, root / "DeepMzyme_Data" / dataset_name, root]
    if root.exists():
        candidates.extend(path for path in root.rglob(dataset_name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate.resolve()
    return None


def find_site_summary_csv(split_dir: Path, split_name: str) -> Path:
    preferred_names = [
        "final_data_summarazing_table_transition_metals_only_catalytic.csv",
        "catalytic_only_summary.csv",
    ]
    candidates = [split_dir / name for name in preferred_names]
    candidates.extend(sorted(split_dir.glob("*.csv")))
    for candidate in candidates:
        if candidate.exists() and is_site_summary_csv(candidate):
            return candidate.resolve()
    existing = [(str(path), classify_summary_csv(path)) for path in candidates if path.exists()]
    raise FileNotFoundError(
        f"No site-level MAHOMES summary CSV found for {split_name} under {split_dir}. "
        f"Existing CSV candidates and detected types: {existing}"
    )


def structure_files(path: Path) -> list[Path]:
    files = []
    for pattern in STRUCTURE_EXTENSIONS:
        files.extend(path.glob(pattern))
    return sorted(files)


DATASET_ROOT = None
for root in DATA_ROOT_CANDIDATES:
    DATASET_ROOT = find_dataset_root(Path(root), CONFIG["data"]["dataset_name"])
    if DATASET_ROOT is not None:
        break
if DATASET_ROOT is None:
    raise FileNotFoundError(
        f"Could not find {CONFIG['data']['dataset_name']!r} under {[str(path) for path in DATA_ROOT_CANDIDATES]}"
    )

TRAIN_DIR = Path(CONFIG["data"]["train_dir_override"]).expanduser().resolve() if CONFIG["data"]["train_dir_override"].strip() else DATASET_ROOT / "train"
TEST_DIR = Path(CONFIG["data"]["test_dir_override"]).expanduser().resolve() if CONFIG["data"]["test_dir_override"].strip() else DATASET_ROOT / "test"
TRAIN_CSV = Path(CONFIG["data"]["train_csv_override"]).expanduser().resolve() if CONFIG["data"]["train_csv_override"].strip() else find_site_summary_csv(TRAIN_DIR, "train")
TEST_CSV = Path(CONFIG["data"]["test_csv_override"]).expanduser().resolve() if CONFIG["data"]["test_csv_override"].strip() else find_site_summary_csv(TEST_DIR, "test")

for name, path in [("TRAIN_DIR", TRAIN_DIR), ("TEST_DIR", TEST_DIR)]:
    if not path.is_dir():
        raise FileNotFoundError(f"{name} does not exist or is not a directory: {path}")
for name, path in [("TRAIN_CSV", TRAIN_CSV), ("TEST_CSV", TEST_CSV)]:
    if not path.is_file():
        raise FileNotFoundError(f"{name} does not exist: {path}")
    detected = classify_summary_csv(path)
    if detected != "site-level MAHOMES summary CSV":
        raise ValueError(f"{name} must be a site-level MAHOMES summary CSV, got {detected}: {path}")

TRAIN_STRUCTURES = structure_files(TRAIN_DIR)
TEST_STRUCTURES = structure_files(TEST_DIR)
if not TRAIN_STRUCTURES:
    raise FileNotFoundError(f"No structure files found in TRAIN_DIR: {TRAIN_DIR}")
if not TEST_STRUCTURES:
    raise FileNotFoundError(f"No structure files found in TEST_DIR: {TEST_DIR}")

DATA_ROOT = DATASET_ROOT.parent
print("TRAIN_DIR:", TRAIN_DIR)
print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_DIR:", TEST_DIR)
print("TEST_CSV:", TEST_CSV)
print("Number of train structures:", len(TRAIN_STRUCTURES))
print("Number of test structures:", len(TEST_STRUCTURES))
print("Train CSV rows:", count_csv_rows(TRAIN_CSV))
print("Test CSV rows:", count_csv_rows(TEST_CSV))
print("Detected train CSV type:", classify_summary_csv(TRAIN_CSV))
print("Detected test CSV type:", classify_summary_csv(TEST_CSV))
print("Why this CSV is used: it has site-level MAHOMES label columns required by src/train.py; structure-level inspection CSVs with semicolon-joined multi-metal labels are not accepted for single-label training.")

In [ ]:
#@title Build planned configuration commands
import hashlib
import itertools
import json
import os
import re
import shlex
import sys
from pathlib import Path

try:
    import pandas as pd
    from IPython.display import display
except Exception:
    pd = None
    display = None

from data_structures import NODE_FEATURES_BY_SET, validate_node_feature_omissions
from training.config import build_arg_parser

TRAIN_HELP_FLAGS = {action.option_strings[0] for action in build_arg_parser()._actions if action.option_strings}

TASK_SELECTION_DEFAULTS = {
    "metal": "val_metal_balanced_acc",
    "ec": "val_ec_group_balanced_acc",
    "joint": "val_joint_balanced_acc",
}
MODEL_PRESET_MAP = {
    "Only-GVP": {"model_architecture": "only_gvp", "fusion_mode": None, "uses_esm": False, "use_early_esm": False},
    "Only-ESM": {"model_architecture": "only_esm", "fusion_mode": None, "uses_esm": True, "use_early_esm": False},
    "GVP + late fusion": {"model_architecture": "gvp", "fusion_mode": "late_fusion", "uses_esm": True, "use_early_esm": False},
    "GVP + early fusion": {"model_architecture": "gvp", "fusion_mode": "early_fusion", "uses_esm": True, "use_early_esm": True},
    "GVP + node-level late fusion": {"model_architecture": "gvp", "fusion_mode": "node_level_late_fusion", "uses_esm": True, "use_early_esm": False},
    "GVP + hybrid fusion": {"model_architecture": "gvp", "fusion_mode": "hybrid", "uses_esm": True, "use_early_esm": True},
    "GVP + cross-modal attention": {"model_architecture": "gvp", "fusion_mode": "cross_modal_attention", "uses_esm": True, "use_early_esm": False},
    "SimpleGNN + ESM": {"model_architecture": "simple_gnn_esm", "fusion_mode": "late_fusion", "uses_esm": True, "use_early_esm": False},
}
RING_MODE_TAGS = {
    "radius_only": "radius_only",
    "radius_plus_precomputed_ring": "precomputed_ring",
    "generate_missing_ring": "generate_ring",
}
VALID_RING_MODES = set(RING_MODE_TAGS)
RING_FILE_PATTERNS = ("*ringEdges", "*.ringEdges", "*_ring_edges.pt", "*_ring_edges.json", "*.pt")


def split_csv(text: str) -> list[str]:
    return [token.strip() for token in str(text or "").split(",") if token.strip()]


def parse_strings(text: str, name: str, allowed: set[str] | None = None) -> list[str]:
    values = split_csv(text)
    if not values:
        raise ValueError(f"{name} must contain at least one value.")
    if allowed is not None:
        bad = [value for value in values if value not in allowed]
        if bad:
            raise ValueError(f"Unsupported {name} value(s): {bad}. Allowed: {sorted(allowed)}")
    return values


def parse_ints(text: str, name: str) -> list[int]:
    values = []
    for token in split_csv(text):
        try:
            values.append(int(token))
        except ValueError as exc:
            raise ValueError(f"{name} must contain comma-separated integers; bad token {token!r}.") from exc
    if not values:
        raise ValueError(f"{name} must contain at least one value.")
    return values


def parse_floats(text: str, name: str) -> list[float]:
    values = []
    for token in split_csv(text):
        try:
            values.append(float(token))
        except ValueError as exc:
            raise ValueError(f"{name} must contain comma-separated numbers; bad token {token!r}.") from exc
    if not values:
        raise ValueError(f"{name} must contain at least one value.")
    return values


def optional_int(text: str, name: str) -> int | None:
    text = str(text or "").strip()
    if not text or text.lower() == "none":
        return None
    try:
        return int(text)
    except ValueError as exc:
        raise ValueError(f"{name} must be blank or an integer, got {text!r}.") from exc


def slugify(value: object) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9.+]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "value"


def format_float_for_name(value: float) -> str:
    return f"{value:g}"


def count_matching_files(path: Path, patterns: tuple[str, ...]) -> int:
    if not path or not path.exists():
        return 0
    found = set()
    for pattern in patterns:
        found.update(candidate for candidate in path.rglob(pattern) if candidate.is_file())
    return len(found)


def resolve_path(value: str, *, default: Path | None = None, must_exist: bool = False) -> Path:
    text = str(value or "").strip()
    if not text:
        if default is None:
            return Path("")
        return default.expanduser().resolve() if default.exists() else default
    path = Path(text).expanduser()
    if path.is_absolute():
        resolved = path
    else:
        candidates = [REPO_DIR / path, DATA_ROOT / path, DATA_ROOT.parent / path, Path("/content") / path, DRIVE_DATA_DIR / path]
        if str(path).startswith("DeepMzyme_Data/"):
            candidates.append(DATA_ROOT / str(path).removeprefix("DeepMzyme_Data/"))
        resolved = next((candidate.resolve() for candidate in candidates if candidate.exists()), (REPO_DIR / path).resolve())
    if must_exist and not resolved.exists():
        raise FileNotFoundError(f"Required path does not exist: {resolved}")
    return resolved


def resolve_esm_embeddings_dir() -> Path:
    candidates = []
    if CONFIG["esm"]["esm_embeddings_dir"].strip():
        return resolve_path(CONFIG["esm"]["esm_embeddings_dir"])
    candidates.extend([
        DATA_ROOT / "esm_embeddings",
        DATA_ROOT / "embeddings",
        REPO_DIR / "DeepMzyme_Data" / "embeddings",
        REPO_DIR / "DeepMzyme_Data" / "esm_embeddings",
    ])
    if CONFIG["data"]["colab_data_source"] == "drive":
        candidates.extend([DRIVE_DATA_DIR / "esm_embeddings", DRIVE_DATA_DIR / "embeddings"])
    return next((candidate.resolve() for candidate in candidates if candidate.exists()), candidates[0])


def resolve_ring_features_dir() -> Path:
    default = DATA_ROOT / "RING_features"
    if CONFIG["ring"]["ring_features_dir"].strip():
        return resolve_path(CONFIG["ring"]["ring_features_dir"])
    if CONFIG["data"]["colab_data_source"] == "drive" and (DRIVE_DATA_DIR / "RING_features").exists():
        return (DRIVE_DATA_DIR / "RING_features").resolve()
    return default.resolve() if default.exists() else default


def parse_omit_node_feature_sets(text: str, valid_features: list[str]) -> list[tuple[str, ...]]:
    if str(text or "") == "":
        return [()]
    parsed = []
    for entry in str(text).split(";"):
        features = tuple(token.strip() for token in entry.split(",") if token.strip())
        validate_node_feature_omissions("conservative", features)
        parsed.append(features)
    return parsed or [()]


def omission_tag(features: tuple[str, ...]) -> str:
    if not features:
        return "fullfeatures"
    if set(features) == {"v_cb_to_fg", "v_res_to_metal"}:
        return "omit_vecgeom"
    body = "_".join(slugify(feature) for feature in features)
    if len(body) > 48:
        digest = hashlib.sha1(",".join(features).encode("utf-8")).hexdigest()[:8]
        body = body[:39].rstrip("_") + "_" + digest
    return "omit_" + body


def config_digest(config: dict[str, object]) -> str:
    payload = json.dumps({key: str(value) for key, value in sorted(config.items()) if key not in {"command", "env", "shell_command"}}, sort_keys=True)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:8]


def make_run_name(config: dict[str, object]) -> str:
    prefix = CONFIG["output"]["run_name_prefix"].strip()
    parts = []
    if prefix:
        parts.append(slugify(prefix))
    parts.extend([
        slugify(config["model_preset"]),
        str(config["task"]),
        RING_MODE_TAGS[str(config["ring_mode"])],
        str(config["omit_tag"]),
        "lr" + format_float_for_name(float(config["learning_rate"])),
        "seed" + str(config["seed"]),
    ])
    name = "_".join(parts)
    if len(name) > 120:
        name = name[:111].rstrip("_") + "_" + config_digest(config)
    return name


def add_flag(cmd: list[object], flag: str, value: object | None = None) -> None:
    if flag not in TRAIN_HELP_FLAGS:
        raise ValueError(f"Notebook tried to use unsupported src/train.py flag: {flag}")
    cmd.append(flag)
    if value is not None:
        cmd.append(value)


def command_with_env(config: dict[str, object]) -> str:
    prefix = []
    env = config.get("env", {})
    if isinstance(env, dict):
        for key in sorted(env):
            value = str(env[key])
            if value:
                prefix.append(f"{key}={shlex.quote(value)}")
    return " ".join(prefix + [shlex.join([str(part) for part in config["command"]])])


def build_train_command(config: dict[str, object]) -> list[object]:
    preset = MODEL_PRESET_MAP[str(config["model_preset"])]
    cmd = [sys.executable, str(REPO_DIR / "src" / "train.py")]
    add_flag(cmd, "--task", config["task"])
    add_flag(cmd, "--structure-dir", TRAIN_DIR)
    add_flag(cmd, "--summary-csv", TRAIN_CSV)
    add_flag(cmd, "--runs-dir", config["runs_dir"])
    add_flag(cmd, "--run-name", config["run_name"])
    add_flag(cmd, "--model-architecture", preset["model_architecture"])
    add_flag(cmd, "--epochs", config["epochs"])
    add_flag(cmd, "--batch-size", config["batch_size"])
    add_flag(cmd, "--learning-rate", config["learning_rate"])
    add_flag(cmd, "--weight-decay", config["weight_decay"])
    add_flag(cmd, "--seed", config["seed"])
    add_flag(cmd, "--val-fraction", config["val_fraction"])
    add_flag(cmd, "--split-by", config["split_by"])
    add_flag(cmd, "--selection-metric", config["selection_metric"])
    add_flag(cmd, "--device", config["device"])
    add_flag(cmd, "--node-feature-set", "conservative")
    add_flag(cmd, "--hidden-s", config["hidden_s"])
    add_flag(cmd, "--hidden-v", config["hidden_v"])
    add_flag(cmd, "--edge-hidden", config["edge_hidden"])
    add_flag(cmd, "--gvp-layers", config["gvp_layers"])
    add_flag(cmd, "--edge-radius", config["edge_radius"])
    add_flag(cmd, "--esm-fusion-dim", config["esm_fusion_dim"])
    add_flag(cmd, "--head-mlp-layers", config["head_mlp_layers"])
    add_flag(cmd, "--esm-dim", CONFIG["esm"]["esm_dim"])
    add_flag(cmd, "--node-rbf-sigma", CONFIG["advanced"]["node_rbf_sigma"])
    add_flag(cmd, "--edge-rbf-sigma", CONFIG["advanced"]["edge_rbf_sigma"])
    add_flag(cmd, "--external-feature-source", CONFIG["advanced"]["external_feature_source"])
    add_flag(cmd, "--metal-loss-function", CONFIG["advanced"]["metal_loss_function"])
    add_flag(cmd, "--metal-focal-gamma", CONFIG["advanced"]["metal_focal_gamma"])
    add_flag(cmd, "--metal-label-smoothing", CONFIG["advanced"]["metal_label_smoothing"])
    add_flag(cmd, "--metal-loss-weight", CONFIG["advanced"]["metal_loss_weight"])
    add_flag(cmd, "--ec-loss-weight", CONFIG["advanced"]["ec_loss_weight"])
    add_flag(cmd, "--mn-loss-multiplier", CONFIG["advanced"]["mn_loss_multiplier"])
    add_flag(cmd, "--cu-loss-multiplier", CONFIG["advanced"]["cu_loss_multiplier"])
    add_flag(cmd, "--zn-loss-multiplier", CONFIG["advanced"]["zn_loss_multiplier"])
    add_flag(cmd, "--fe-loss-multiplier", CONFIG["advanced"]["fe_loss_multiplier"])
    add_flag(cmd, "--co-loss-multiplier", CONFIG["advanced"]["co_loss_multiplier"])
    add_flag(cmd, "--ni-loss-multiplier", CONFIG["advanced"]["ni_loss_multiplier"])
    add_flag(cmd, "--class-viii-loss-multiplier", CONFIG["advanced"]["class_viii_loss_multiplier"])
    add_flag(cmd, "--unsupported-metal-policy", CONFIG["advanced"]["unsupported_metal_policy"])
    add_flag(cmd, "--invalid-structure-policy", CONFIG["advanced"]["invalid_structure_policy"])
    add_flag(cmd, "--lr-schedule", config["lr_schedule"])
    if str(config["lr_schedule"]) == "step":
        add_flag(cmd, "--lr-step-size", CONFIG["advanced"]["lr_step_size"])
        add_flag(cmd, "--lr-decay-gamma", CONFIG["advanced"]["lr_decay_gamma"])
    if CONFIG["advanced"]["node_rbf_use_raw_distances"]:
        add_flag(cmd, "--node-rbf-use-raw-distances")
    if CONFIG["advanced"]["deterministic"]:
        add_flag(cmd, "--deterministic")
    if CONFIG["advanced"]["save_epoch_checkpoints"]:
        add_flag(cmd, "--save-epoch-checkpoints")
    if CONFIG["advanced"]["allow_train_loss_test_eval_debug"]:
        add_flag(cmd, "--allow-train-loss-test-eval-debug")
    if CONFIG["advanced"]["balance_metal_site_symbols"]:
        add_flag(cmd, "--balance-metal-site-symbols")
    if CONFIG["advanced"]["require_all_task_classes"]:
        add_flag(cmd, "--require-all-task-classes")
    if CONFIG["advanced"]["allow_missing_external_features"]:
        add_flag(cmd, "--allow-missing-external-features")
    if CONFIG["advanced"]["external_features_root_dir"].strip():
        add_flag(cmd, "--external-features-root-dir", resolve_path(CONFIG["advanced"]["external_features_root_dir"]))
    if config["n_folds"] is not None:
        add_flag(cmd, "--n-folds", config["n_folds"])
        add_flag(cmd, "--fold-index", config["fold_index"])
    if config["task"] in {"ec", "joint"}:
        add_flag(cmd, "--ec-label-depth", config["ec_label_depth"])
        add_flag(cmd, "--ec-group-weighting", CONFIG["advanced"]["ec_group_weighting"])
        add_flag(cmd, "--ec-contrastive-weight", config["ec_contrastive_weight"])
        add_flag(cmd, "--ec-contrastive-temperature", CONFIG["advanced"]["ec_contrastive_temperature"])
    if config["omit_node_features"]:
        add_flag(cmd, "--omit-node-features", ",".join(config["omit_node_features"]))
    if CONFIG["esm"]["disable_esm_branch"]:
        add_flag(cmd, "--disable-esm-branch")
    if preset["fusion_mode"]:
        add_flag(cmd, "--fusion-mode", preset["fusion_mode"])
    if preset["use_early_esm"]:
        add_flag(cmd, "--use-early-esm")
        add_flag(cmd, "--early-esm-dim", CONFIG["advanced"]["early_esm_dim"])
        add_flag(cmd, "--early-esm-dropout", CONFIG["advanced"]["early_esm_dropout"])
        add_flag(cmd, "--early-esm-scope", CONFIG["advanced"]["early_esm_scope"])
        if CONFIG["advanced"]["early_esm_raw"]:
            add_flag(cmd, "--early-esm-raw")
    if preset["fusion_mode"] == "cross_modal_attention":
        add_flag(cmd, "--cross-attention-layers", config["cross_attention_layers"])
        add_flag(cmd, "--cross-attention-heads", config["cross_attention_heads"])
        add_flag(cmd, "--cross-attention-dropout", CONFIG["advanced"]["cross_attention_dropout"])
        add_flag(cmd, "--cross-attention-neighborhood", CONFIG["advanced"]["cross_attention_neighborhood"])
        if CONFIG["advanced"]["cross_attention_bidirectional"]:
            add_flag(cmd, "--cross-attention-bidirectional")
    if preset["uses_esm"]:
        add_flag(cmd, "--esm-embeddings-dir", config["esm_embeddings_dir"])
        if CONFIG["esm"]["allow_missing_esm_embeddings"]:
            add_flag(cmd, "--allow-missing-esm-embeddings")
        if not CONFIG["esm"]["prepare_missing_esm_embeddings"]:
            add_flag(cmd, "--no-prepare-missing-esm-embeddings")
    else:
        add_flag(cmd, "--no-prepare-missing-esm-embeddings")
    if config["ring_mode"] == "radius_only":
        add_flag(cmd, "--no-prepare-missing-ring-edges")
    elif config["ring_mode"] == "radius_plus_precomputed_ring":
        add_flag(cmd, "--use-ring-edges")
        add_flag(cmd, "--require-ring-edges")
        add_flag(cmd, "--ring-features-dir", config["ring_features_dir"])
        add_flag(cmd, "--no-prepare-missing-ring-edges")
    elif config["ring_mode"] == "generate_missing_ring":
        add_flag(cmd, "--use-ring-edges")
        add_flag(cmd, "--ring-features-dir", config["ring_features_dir"])
        add_flag(cmd, "--prepare-missing-ring-edges")
    else:
        raise ValueError(f"Unsupported ring_mode: {config['ring_mode']}")
    if CONFIG["basic"]["run_held_out_test_eval"]:
        add_flag(cmd, "--test-structure-dir", TEST_DIR)
        add_flag(cmd, "--test-summary-csv", TEST_CSV)
        add_flag(cmd, "--run-test-eval")
    return cmd


valid_omit_features = list(NODE_FEATURES_BY_SET["conservative"])
omit_feature_sets = parse_omit_node_feature_sets(CONFIG["node_features"]["omit_node_feature_sets"], valid_omit_features)
model_presets = parse_strings(CONFIG["configuration_comparison"]["model_presets_csv"], "MODEL_PRESETS_CSV", set(MODEL_PRESET_MAP))
ring_modes = parse_strings(CONFIG["configuration_comparison"]["ring_edge_modes_csv"], "RING_EDGE_MODES_CSV", VALID_RING_MODES)
learning_rates = parse_floats(CONFIG["configuration_comparison"]["learning_rates_csv"], "LEARNING_RATES_CSV")
weight_decays = parse_floats(CONFIG["configuration_comparison"]["weight_decays_csv"], "WEIGHT_DECAYS_CSV")
batch_sizes = parse_ints(CONFIG["configuration_comparison"]["batch_sizes_csv"], "BATCH_SIZES_CSV")
seeds = parse_ints(CONFIG["configuration_comparison"]["seeds_csv"], "SEEDS_CSV")
hidden_s_values = parse_ints(CONFIG["advanced"]["hidden_s_values_csv"], "HIDDEN_S_VALUES_CSV")
hidden_v_values = parse_ints(CONFIG["advanced"]["hidden_v_values_csv"], "HIDDEN_V_VALUES_CSV")
edge_hidden_values = parse_ints(CONFIG["advanced"]["edge_hidden_values_csv"], "EDGE_HIDDEN_VALUES_CSV")
gvp_layer_values = parse_ints(CONFIG["advanced"]["gvp_layers_values_csv"], "GVP_LAYERS_VALUES_CSV")
head_mlp_layer_values = parse_ints(CONFIG["advanced"]["head_mlp_layers_values_csv"], "HEAD_MLP_LAYERS_VALUES_CSV")
edge_radius_values = parse_floats(CONFIG["advanced"]["edge_radius_values_csv"], "EDGE_RADIUS_VALUES_CSV")
esm_fusion_dim_values = parse_ints(CONFIG["advanced"]["esm_fusion_dim_values_csv"], "ESM_FUSION_DIM_VALUES_CSV")
lr_schedules = parse_strings(CONFIG["advanced"]["lr_schedules_csv"], "LR_SCHEDULES_CSV", {"fixed", "cosine", "step"})
ec_label_depths = parse_ints(CONFIG["advanced"]["ec_label_depths_csv"], "EC_LABEL_DEPTHS_CSV")
ec_contrastive_weights = parse_floats(CONFIG["advanced"]["ec_contrastive_weights_csv"], "EC_CONTRASTIVE_WEIGHTS_CSV")
cross_attention_layer_values = parse_ints(CONFIG["advanced"]["cross_attention_layers_csv"], "CROSS_ATTENTION_LAYERS_CSV")
cross_attention_head_values = parse_ints(CONFIG["advanced"]["cross_attention_heads_csv"], "CROSS_ATTENTION_HEADS_CSV")
n_folds_value = optional_int(CONFIG["advanced"]["n_folds"], "N_FOLDS")
fold_index_value = optional_int(CONFIG["advanced"]["fold_index"], "FOLD_INDEX")
if (n_folds_value is None) != (fold_index_value is None):
    raise ValueError("N_FOLDS and FOLD_INDEX must both be blank or both set.")

runs_dir = resolve_path(CONFIG["output"]["runs_dir"], default=(Path("/content/deepmzyme_outputs/runs") if Path("/content").exists() else REPO_DIR / "DeepMzyme_Data" / "notebook_outputs" / "runs"))
runs_dir.mkdir(parents=True, exist_ok=True)
esm_embeddings_dir = resolve_esm_embeddings_dir()
ring_features_dir = resolve_ring_features_dir()
ring_exe_path = resolve_path(CONFIG["ring"]["ring_exe_path"])
esm_file_count = count_matching_files(esm_embeddings_dir, ("*_esmc.pt", "*.pt"))
ring_file_count = count_matching_files(ring_features_dir, RING_FILE_PATTERNS)
selected_presets_require_esm = any(MODEL_PRESET_MAP[preset]["uses_esm"] for preset in model_presets)

print("Valid MODEL_PRESETS_CSV values:", ", ".join(MODEL_PRESET_MAP))
print("Valid RING_EDGE_MODES_CSV values:", ", ".join(sorted(VALID_RING_MODES)))
print("Valid LR_SCHEDULES_CSV values: fixed, cosine, step")
print("Valid TASK values: metal, ec, joint")
print("Valid omit-able node feature names:")
print(", ".join(valid_omit_features))
print("Selected omission settings:", [",".join(features) if features else "full node feature set" for features in omit_feature_sets])
print("Chosen ESM embeddings directory:", esm_embeddings_dir)
print("ESM embeddings directory exists:", esm_embeddings_dir.exists())
print("Number of *_esmc.pt or .pt files found:", esm_file_count)
print("Missing ESM generation enabled:", CONFIG["esm"]["prepare_missing_esm_embeddings"])
print("Missing ESM allowed:", CONFIG["esm"]["allow_missing_esm_embeddings"])
print("Selected model presets require ESM:", selected_presets_require_esm)
print("Original RING_EXE_PATH:", CONFIG["ring"]["ring_exe_path"])
print("Resolved RING_EXE_PATH:", ring_exe_path)
print("RING_EXE_PATH exists:", ring_exe_path.exists())
print("RING_EXE_PATH executable:", os.access(ring_exe_path, os.X_OK) if ring_exe_path else False)
print("RING_FEATURES_DIR:", ring_features_dir)
print("Existing RING edge files:", ring_file_count)

if selected_presets_require_esm and not (esm_embeddings_dir.exists() and esm_file_count > 0):
    message = "ESM-using presets are selected, but no ESM embedding files were found."
    if CONFIG["basic"]["run_training"] and not (CONFIG["esm"]["allow_missing_esm_embeddings"] or CONFIG["esm"]["prepare_missing_esm_embeddings"]):
        raise FileNotFoundError(message + " Set ESM_EMBEDDINGS_DIR, enable PREPARE_MISSING_ESM_EMBEDDINGS, or enable ALLOW_MISSING_ESM_EMBEDDINGS.")
    print("Warning:", message)

if (selected_presets_require_esm
        and CONFIG["esm"]["prepare_missing_esm_embeddings"]
        and not CONFIG["data"]["install_esm_dependencies"]):
    print("Warning: PREPARE_MISSING_ESM_EMBEDDINGS is enabled but INSTALL_ESM_DEPENDENCIES is False."
          " ESM embedding generation will fail at runtime unless the esm package is already installed.")

all_combinations = list(itertools.product(
    model_presets,
    ring_modes,
    omit_feature_sets,
    learning_rates,
    weight_decays,
    batch_sizes,
    seeds,
    hidden_s_values,
    hidden_v_values,
    edge_hidden_values,
    gvp_layer_values,
    head_mlp_layer_values,
    edge_radius_values,
    esm_fusion_dim_values,
    lr_schedules,
    ec_label_depths,
    ec_contrastive_weights,
    cross_attention_layer_values,
    cross_attention_head_values,
))
total_planned_before_limit = len(all_combinations)
max_runs = CONFIG["configuration_comparison"]["max_configuration_runs"]
if max_runs < 1:
    raise ValueError("MAX_CONFIGURATION_RUNS must be at least 1.")
limited_combinations = all_combinations[:max_runs]
if total_planned_before_limit > max_runs:
    print(f"Warning: limiting planned configurations from {total_planned_before_limit} to {max_runs}.")

planned_runs = []
for combo in limited_combinations:
    (
        model_preset,
        ring_mode,
        omit_features,
        learning_rate,
        weight_decay,
        batch_size,
        seed,
        hidden_s,
        hidden_v,
        edge_hidden,
        gvp_layers,
        head_mlp_layers,
        edge_radius,
        esm_fusion_dim,
        lr_schedule,
        ec_label_depth,
        ec_contrastive_weight,
        cross_attention_layers,
        cross_attention_heads,
    ) = combo
    preset = MODEL_PRESET_MAP[model_preset]
    selection_metric = CONFIG["advanced"]["selection_metric"].strip() or TASK_SELECTION_DEFAULTS[CONFIG["basic"]["task"]]
    run_config = {
        "task": CONFIG["basic"]["task"],
        "model_preset": model_preset,
        "model_architecture": preset["model_architecture"],
        "fusion_mode": preset["fusion_mode"] or "none",
        "uses_esm": preset["uses_esm"],
        "ring_mode": ring_mode,
        "ring_features_dir": str(ring_features_dir),
        "ring_exe_path": str(ring_exe_path),
        "omit_node_features": omit_features,
        "omit_tag": omission_tag(omit_features),
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "seed": seed,
        "hidden_s": hidden_s,
        "hidden_v": hidden_v,
        "edge_hidden": edge_hidden,
        "gvp_layers": gvp_layers,
        "head_mlp_layers": head_mlp_layers,
        "edge_radius": edge_radius,
        "esm_fusion_dim": esm_fusion_dim,
        "lr_schedule": lr_schedule,
        "ec_label_depth": ec_label_depth,
        "ec_contrastive_weight": ec_contrastive_weight,
        "cross_attention_layers": cross_attention_layers,
        "cross_attention_heads": cross_attention_heads,
        "epochs": CONFIG["basic"]["epochs"],
        "val_fraction": CONFIG["advanced"]["val_fraction"],
        "split_by": CONFIG["advanced"]["split_by"],
        "selection_metric": selection_metric,
        "device": CONFIG["basic"]["device"],
        "n_folds": n_folds_value,
        "fold_index": fold_index_value,
        "esm_embeddings_dir": str(esm_embeddings_dir),
        "runs_dir": str(runs_dir),
    }
    run_config["run_name"] = make_run_name(run_config)
    run_config["run_dir"] = str(runs_dir / run_config["run_name"])
    run_config["env"] = {
        "PYTHONPATH": str(SRC_DIR) + os.pathsep + os.environ.get("PYTHONPATH", ""),
        "RING_FEATURES_DIR": str(ring_features_dir),
    }
    if ring_mode == "generate_missing_ring":
        run_config["env"]["RING_EXE_PATH"] = str(ring_exe_path)
    run_config["command"] = build_train_command(run_config)
    run_config["shell_command"] = command_with_env(run_config)
    planned_runs.append(run_config)

for config in planned_runs:
    ring_mode = config["ring_mode"]
    if ring_mode == "radius_plus_precomputed_ring" and (not ring_features_dir.exists() or ring_file_count == 0):
        message = f"RING mode {ring_mode} needs existing files in {ring_features_dir}."
        if CONFIG["basic"]["run_training"]:
            raise FileNotFoundError(message)
        print("Warning:", message, "Dry-run mode will still show the command.")
    if ring_mode == "generate_missing_ring" and (not ring_exe_path.is_file() or not os.access(ring_exe_path, os.X_OK)):
        message = f"RING mode {ring_mode} needs an executable RING binary at {ring_exe_path}."
        if CONFIG["basic"]["run_training"]:
            raise FileNotFoundError(message)
        print("Warning:", message, "Dry-run mode will still show the command.")

planned_table = [
    {
        "run_index": index,
        "run_name": config["run_name"],
        "task": config["task"],
        "model_preset": config["model_preset"],
        "model_architecture": config["model_architecture"],
        "fusion_mode": config["fusion_mode"],
        "ring_mode": config["ring_mode"],
        "omit_node_features": ",".join(config["omit_node_features"]) if config["omit_node_features"] else "",
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "weight_decay": config["weight_decay"],
        "seed": config["seed"],
        "hidden_s": config["hidden_s"],
        "hidden_v": config["hidden_v"],
        "edge_hidden": config["edge_hidden"],
        "gvp_layers": config["gvp_layers"],
        "edge_radius": config["edge_radius"],
        "esm_fusion_dim": config["esm_fusion_dim"],
        "selection_metric": config["selection_metric"],
        "run_dir": config["run_dir"],
    }
    for index, config in enumerate(planned_runs, start=1)
]

print("Total number of planned configurations before limiting:", total_planned_before_limit)
print("MAX_CONFIGURATION_RUNS:", max_runs)
print("Final number of commands that will actually be run:", len(planned_runs))
if pd is not None and display is not None:
    display(pd.DataFrame(planned_table))
else:
    print(json.dumps(planned_table, indent=2))

print("Exact shell-safe commands:")
for index, config in enumerate(planned_runs, start=1):
    print(f"[{index}] {config['shell_command']}")

print("RING flags by configuration:")
for index, config in enumerate(planned_runs, start=1):
    flags = [str(part) for part in config["command"] if str(part).startswith("--") and "ring" in str(part)]
    print(f"[{index}] ring_mode={config['ring_mode']} flags={flags}")

In [ ]:
#@title Optional training execution
import os
import subprocess
from pathlib import Path


def stream_command(cmd: list[object], *, cwd: Path, env: dict[str, str]) -> int:
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return process.wait()


def run_environment(config: dict[str, object]) -> dict[str, str]:
    env = os.environ.copy()
    for key, value in config.get("env", {}).items():
        if value:
            env[str(key)] = str(value)
    return env


completed_run_dirs = []
failed_run_dirs = []

if not CONFIG["basic"]["run_training"]:
    print("RUN_TRAINING=False, so no training command was launched.")
else:
    if CONFIG["basic"]["device"] == "cuda":
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("DEVICE is cuda, but torch.cuda.is_available() is False. Enable a GPU runtime or set DEVICE to cpu.")
    print("Starting planned configurations:", len(planned_runs))
    for index, config in enumerate(planned_runs, start=1):
        run_dir = Path(config["run_dir"])
        if CONFIG["configuration_comparison"]["skip_existing_runs"] and any((run_dir / marker).exists() for marker in ("run_config.json", "run_metadata.json")):
            print(f"[{index}/{len(planned_runs)}] Skipping existing run: {run_dir}")
            completed_run_dirs.append(run_dir)
            continue
        print("=" * 80)
        print(f"[{index}/{len(planned_runs)}] {config['run_name']}")
        print(config["shell_command"])
        print("=" * 80)
        return_code = stream_command(config["command"], cwd=REPO_DIR, env=run_environment(config))
        if return_code == 0:
            completed_run_dirs.append(run_dir)
            print("Completed:", run_dir)
        else:
            failed_run_dirs.append(run_dir)
            print("Failed with exit code", return_code, ":", run_dir)
            if CONFIG["configuration_comparison"]["stop_on_first_failure"]:
                break

print("Completed run directories:", [str(path) for path in completed_run_dirs])
print("Failed run directories:", [str(path) for path in failed_run_dirs])

In [ ]:
#@title Summarize/report results
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

summary_csv = Path(runs_dir).parent / f"{CONFIG['output']['summary_basename']}.csv"
summary_png = Path(runs_dir).parent / f"{CONFIG['output']['summary_basename']}.png"
run_markers = {"run_config.json", "run_metadata.json", "dataset_summary.json", "test_report.json"}
run_dirs_found = [path for path in sorted(Path(runs_dir).iterdir()) if path.is_dir() and any((path / marker).exists() for marker in run_markers)] if Path(runs_dir).exists() else []

if not run_dirs_found:
    print("No completed run directories found yet:", runs_dir)
else:
    report_cmd = [
        sys.executable,
        str(REPO_DIR / "src" / "report_runs.py"),
        "--runs-dir",
        str(runs_dir),
        "--out-csv",
        str(summary_csv),
        "--out-figure",
        str(summary_png),
    ]
    print(shlex.join([str(part) for part in report_cmd]))
    report_env = os.environ.copy()
    report_env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + report_env.get("PYTHONPATH", "")
    subprocess.run(report_cmd, cwd=REPO_DIR, env=report_env, check=True)
    print("Summary CSV:", summary_csv)
    print("Summary figure:", summary_png)
    try:
        import pandas as pd
        from IPython.display import Image, display
        summary_df = pd.read_csv(summary_csv)
        display(summary_df)
        if summary_png.exists():
            display(Image(filename=str(summary_png)))
    except Exception as exc:
        print("Notebook display skipped:", exc)

if CONFIG["output"]["copy_outputs_to_drive"]:
    if not Path("/content/drive/MyDrive").exists() and Path("/content").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    target_dir = DRIVE_ROOT_PATH / "notebook_outputs"
    target_dir.mkdir(parents=True, exist_ok=True)
    target_runs = target_dir / "runs"
    target_runs.mkdir(parents=True, exist_ok=True)
    for run_dir in run_dirs_found:
        destination = target_runs / run_dir.name
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(run_dir, destination)
    for artifact in (summary_csv, summary_png):
        if artifact.exists():
            shutil.copy2(artifact, target_dir / artifact.name)
    print("Copied outputs to:", target_dir)
else:
    print("Drive copy skipped. Outputs remain under:", Path(runs_dir).parent)

In [ ]:
#@title Optional final held-out test evaluation
import os
import shlex
import subprocess
import sys
from pathlib import Path

RUN_FINAL_HELD_OUT_TEST_EVAL = False  #@param {type:"boolean"}
FINAL_RUN_DIR = ""  #@param {type:"string"}

if not RUN_FINAL_HELD_OUT_TEST_EVAL:
    print("Final held-out test evaluation is off. Enable it only after choosing a configuration by validation metrics.")
else:
    selected_run_dir = Path(FINAL_RUN_DIR).expanduser().resolve() if FINAL_RUN_DIR.strip() else None
    if selected_run_dir is None:
        raise ValueError("Set FINAL_RUN_DIR to the validation-selected run directory before final held-out test evaluation.")
    print("Selected run directory:", selected_run_dir)
    print("This notebook keeps held-out test evaluation separate from configuration comparison. Use the training CLI checkpoint/test options directly if a rerun is needed for final reporting.")
    print("Existing run reports can be summarized with src/report_runs.py; no automatic model choice is made here.")